# Preconditioning experiements: $\alpha$ circulant matrix

This notebook will explore the applications of the methods described in [the ParaDiag](https://arxiv.org/abs/2005.09158) paper. 

## Topelitz and circulant matrices

The wave equation can be written as

$$A\mathbf{u} = \mathbf{s},$$

where $\mathbf{u}$ is the time-space pressure vector, $\mathbf{s}$ is the source term vector, and $A$ is the discrete wave equation operator,

$$A \equiv D_{tt} \otimes I_{xy} + I_{t} \otimes D_{\nabla^2}.$$

Here, $D_{tt} \equiv u_{tt}$ is the discrete operator encoding the second temporal derivative, and $D_{\nabla^2} \equiv c^2\nabla^2$ is the discrete operator encoding the Laplacian. The former is lifted to the full space time domain via the Kroncker product with the identity $I_{xy} \in \mathbb{R}^{n_x n_y \times n_x n_y}$ and the latter over time by $I_{t} \in \mathbb{R}^{n_t \times n_t}$.

The issue with this is that we want to solve $A \delta u = -r$ as a Gauss-Newton subproblem, but the conditioning number of this system is huge and increases superlinearly with time (see [this notebook](./conditioning-study.ipynb)). 

Our opertor matrix has a block Toeplitz structure, meaning all diagonals are the same, by construction. A Toeplitz matrix looks like

$$
A = \begin{bmatrix}
a_0 & a_1 &\cdots & a_{n-2} & a_{n-1} \\
a_1 & a_0 & a_1 & \cdots & a_{n-2} \\
\vdots & \ddots & \ddots & \ddots & \vdots \\
a_{n-2} & & &  & a_1\\
a_{n-1} & & \cdots & a_1 & a_0 \\
\end{bmatrix}
$$

A circulant matrix is very similar, just the off diagonals reappear in the corners:

$$
C = \begin{bmatrix}
c_0 & c_1 &\cdots & c_{n-2} & c_1 \\
c_1 & c_0 & c_1 & \cdots & c_2 \\
\vdots & \ddots & \ddots & \ddots & \vdots \\
c_2 & & &  & c_1\\
c_1 & & \cdots & c_1 & c_0 \\
\end{bmatrix}
$$

A Toeplitz matrix can be transformed into a circulnat matrix by multiplication against a cyclic shift matrix

$$P = \begin{bmatrix}
0 & 1 & 0 & 0 & \cdots & 0 \\
0 & 0 & 1 & 0 & \cdots & 0  \\
\\
\vdots & \ddots & \ddots & \ddots & \vdots \\
0 & & &  & & 1\\
1 & 0 & 0 & \cdots & 0 & 0 \\
\end{bmatrix}$$

## Circulant matrices and Fourier modes

The following discussion follows [this derivation](https://mwilensky768.github.io/assets/memos/Circulant_Matrices_are_diagonalized_by_Fourier_Modes.pdf).

A nice property of circulant matrices is that they are diagonalisable by Fourier modes. A Fourier mode is a sinusoidal component of a wave, representing a single frequency of oscillation that can be combned with the other modes to reconstruct the original function. For our $N \times N$ circulant matrix $C$ we can write

$$C = F\Lambda F^{-1},$$

where the $j, k$-th entry of our Fourier mode matrix is

$$F_{j, k} = \frac 1 {\sqrt{N}} e^{2\pi i \frac {jk}{N}},$$

and $\Lambda$ is a diagonal matrix containing the eigenvalues. 

This means that

$$F^{-1}_{j, k} = \frac 1 {\sqrt N} e^{-2\pi i \frac{jk}{N}}.$$

Thus, we can write that

$$C_{j, l} = \sum_k \lambda_k \frac{1}{N} e^{2\pi i \frac{(j - l)k}{N}}.$$

### Eigenvectors of the cylic shift matrix

Consider again the cyclic shift matrix. Say we have a vector $v$ such that

$$v = \begin{bmatrix}
1 \\ z \\ z^2 \\ \vdots\\ z^{n-1}
\end{bmatrix}.$$

If we apply $P$ then we have

$$Pv = \begin{bmatrix}
z \\ z^2 \\ \vdots\\ z^{n-1}\\ 1
\end{bmatrix}.$$

For this to be an eigenvector we require

$$Pv = \lambda v = \begin{bmatrix}
\lambda \\ \lambda z \\ \lambda z^2 \\ \vdots\\ \lambda z^{n-1}
\end{bmatrix}.$$

Comparing entries we see that

$$z = \lambda,\\ z^2 = \lambda z,$$

and so

$$z^n = 1,$$

so the solutions are

$$z = \omega^k, \text{   where   } \omega = e^{\frac{-2\pi i}{n}},$$

by the Euler identity. Thus, the Fourier vectors $v$ are the eigenvectors of the cyclic shift matrix.

### Eigenvalues of a circulant matrix

If we consider for a moment that the circulant matrix $C$ is a bunch of column vectors $c_1, c_2, \dots, c_{n-1}$, then we can write $C$ as

$$C = c_0 I + c_1 P + c_2 P^2 + \cdots + c_{n-1}P^{n-1},$$

so $C$ is merely a polynomial in $P$.

We have established that

$$P v_k = \omega^k v_k,$$

so 

$$P^2 v_k = \omega^{2k}v_k,$$

and so forth. Therefore

$$Cv_k = (c_0 + c_1\omega^{k} + c_2\omega^{2k} + \ddots)v_k.$$

The Fourier vector is unchanged, but the scalar multiplier has changed. These are the eigenvalues of the circulant matrix.

### Link to the Fourier transform

The eigenvalue of the $k$-th columns vector of the circuant matrix is

$$\lambda_k = c_0 + c_1 \omega^{k} + \cdots + c_{n-1}\omega^{(n-1)k} \equiv \sum^{n-1}_{j=0} c_j \omega^{jk}.$$

The discrete Fourier transform of of the first column

$$c = \begin{bmatrix}
c_0 \\ c_1 \\ \vdots \\ c_{n-1}\\
\end{bmatrix}.$$

The discrete Fourier transform is defined to be

$$F[c_k] = \hat{c}_k = \sum^{n-1}_{j=0} c_j e^{-2\pi i \frac{jk}{N}} \equiv \sum^{n-1}_{j=0} c_j \omega^{jk}.$$

Therefore,

$$F[c_k] \equiv \lambda_k,$$

which is to say that the diagonal eigenvalue matrix $\Lambda$ of a circulant matrix can be built by means of a Fourier transform of the columns of $C$,

$$\Lambda = \text{diag}(F[c_k]).$$

So, a Fast Fourier Transform is an efficient way to figure out the eigenvalues of a circulant matrix.

## Diagonalisation

Since we can efficiently figure out the eigenvalues of a circulant matrix, we can easily diagonalise it. We care to do this because the inverse of a diagonal matrix is trivial to compute - it is merely the reciprocals of the diagonals.

If we collect all of the Fourier vectors (the $z = \omega^k$) into a matrix

$$F = \begin{bmatrix}
f_0 & \cdots & f_{n-1}
\end{bmatrix},$$

then

$$CF = \begin{bmatrix}
Cf_0 & Cf_{1} & \cdots
\end{bmatrix}.$$

But since each column satisfies

$$Cf_k = \lambda_k f_k,$$

as we showed above then

$$CF = \begin{bmatrix}
\lambda_0f_0 & \lambda_1f_{1} & \cdots
\end{bmatrix} = 
F\begin{bmatrix}
\lambda_0 & &\\
& \lambda_1 &\\
& & \ddots
\end{bmatrix}.$$

Hence

$$CF = F\Lambda,$$

and 

$$C = F\Lambda F^{-1}.$$

Thus, a circulant matrix can be easily diagonalised by an FFT. What this equation means is that by applying the change of basis to $C$

$$F^{-1}CF \Rightarrow \Lambda,$$

then we are representing the linear operator as a diagonal matrix in a Fourier basis.

## Computing the inverse of a circulant matrix

We have shown that in a Fourier basis we can represent a circulant matrix by a diagonal matrix whose entries are the eigenvalues of $C$. We have alos shown that these eigenvalues are easy to compute for each column $c$ by

$$\lambda_k = c_0 + c_1 \omega^{k} + \cdots + c_{n-1}\omega^{k(n-1)} = \sum^{n-1}_{j=0} c_j \omega^{jk},$$

where,

$$\omega = e^{\frac{-2\pi i}{n}}$$

and then $\omega^{k}$ is the eigenvalue $k$-th  

Thus, we can easily find a diagonal representation of $C$ in the Fourier basis. 

Matrix inversion in the normal basis corresponds to division in the Fourier basis, so

$$\Lambda^{-1} = \begin{bmatrix}
\frac 1 {\lambda_0} & & \\[2pt]
& \frac 1 {\lambda_1} &\\[2pt]
& & \ddots\\
\end{bmatrix},$$

is exactly the inverse of $C$ in the Fourier basis.

### Practicality

As a result of the above, applying $C^{-1}$ to a vector can be done efficiently by

i. transforming to Fourier space by FFT

ii. scaling each Fourier coefficient by $1/\lambda_k$ and

iii. transforming back

## Application to our problem

As discussed, our linear system is poorly conditioned. A good preconditioner $M$ is one such that

$$M^{-1}A x = M^{-1}b$$

is well conditioned, where $M \approx A.$

In our circumstance, $A$ is nearly circulant, being a block Toeplitz matrix. In particular, it is what is referred to as Block Toeplitz wiht Toeplitz Blocks (BTTB). 

Thus, a circulant $M$ is a decent approximation of $A$, missing only the corner bondary values, and we can easily compute its inverse via the FFT method applied above. This is the basis of the circulant preconditioning method. Physically, by approximating $A$ as circulant, we are replacing physical boundary conditions with periodic ones.

A circulant preconditioner satisfies

$$M = F \Lambda F^{-1},$$

so, in our case of the inner GN solve,

$$M\delta u = -r$$

becomes

$$\delta u = -F \Lambda^{-1}F^{-1}r.$$

Then, from left to right the process reads "to compute $\delta u,$" transform our circulant approximation to a Fourier basis by means of a multplication by the matrix of Fourier modes $F$, apply the diagonal approximation of $A^{-1}, \ \Lambda^{-1}$, and transform back via $F^{-1}$.

## The ParaDiag family

The ParaDiag family of preconditioners use what are called $\alpha$ circulant matrices. The idea is that we approximate the constituent operators of $A$ by circulant forms,

$$M_\alpha = C_\alpha(D_{tt}) \otimes I_{xy} + I_t \otimes D_{\nabla^2}$$

where 

$$C_\alpha = C_α = c_0I + c_1P_\alpha + c_2P_\alpha² + \cdots \\=\begin{bmatrix} c_0 & c_1 & c_2 & \alpha c_3 \ \alpha c_3 & c_0 & c_1 & c_2 \ \alpha c_2 & \alpha c_3 & c_0 & c_1 \ \alpha c_1 & \alpha c_2 & \alpha c_3 & c_0 \end{bmatrix} \quad (n = 4 \text{ example})$$

Here, $\alpha \in [0, 1)$ is a free parameter. This parameter is introduced to help apply the block circulant preconditioning techique to hyperbolic problems, such as the wave equation. When $\alpha = 1$, the circulant matrix is perfectly periodic, but this is the wrong assumption for hyperbolic problems. Introducing $\alpha$ allows us to vary the wrap around behaviour and make it more appropriate for hyperbolic probems.

## A toy scalar example

